# 📚 SQL Ch.5 — Conditions & NULL Handling
> BigQuery SQL Reference Guide, Chapter 5: CASE WHEN · IS NULL · COALESCE · NULLIF · BETWEEN/IN · SAFE_DIVIDE/SAFE_CAST · COUNTIF/IF  
> BigQuery SQL 완전 참조 가이드 5장: CASE WHEN · IS NULL · COALESCE · NULLIF · BETWEEN/IN · SAFE_DIVIDE/SAFE_CAST · COUNTIF/IF

---
# 🎯 Learning Objective
Today I want to learn: / 오늘 배우고 싶은 것:
- [x] Branch a query's output on a condition with `CASE WHEN`, from a simple 2-way split to a multi-tier classification  
`CASE WHEN`으로 단순 2분기부터 다단계 등급 분류까지 조건에 따라 결과를 분기한다
- [x] Explain why `NULL` needs `IS NULL`/`IS NOT NULL` instead of `=`, and use `COALESCE`/`NULLIF` to handle it safely  
`NULL`이 왜 `=` 대신 `IS NULL`/`IS NOT NULL`이 필요한지 설명하고 `COALESCE`/`NULLIF`로 안전하게 다룬다
- [x] Use `SAFE_DIVIDE`, `SAFE_CAST`, and `COUNTIF` to write queries that don't break when the data is messy  
`SAFE_DIVIDE`, `SAFE_CAST`, `COUNTIF`로 데이터가 지저분해도 깨지지 않는 쿼리를 작성한다

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

**EN:** This chapter is about two related survival skills: branching logic (`CASE WHEN` — "do this if X, that if Y") and handling missing data (`NULL`, SQL's special "unknown" marker, which breaks ordinary comparisons like `=` in ways that surprise beginners). Functions like `COALESCE`, `NULLIF`, `SAFE_DIVIDE`, and `SAFE_CAST` exist specifically to make `NULL` — and the errors it can cause — controllable instead of dangerous.

**KR:** 이번 챕터는 서로 연결된 두 가지 생존 기술을 다룹니다: 분기 로직(`CASE WHEN` — "X면 이거, Y면 저거")과 결측 데이터 다루기(`NULL`, SQL의 특수한 "알 수 없음" 표시인데, 초보자를 놀라게 하는 방식으로 `=` 같은 일반 비교를 깨뜨립니다). `COALESCE`, `NULLIF`, `SAFE_DIVIDE`, `SAFE_CAST` 같은 함수들은 `NULL`과 그것이 일으킬 수 있는 오류를 위험한 것이 아니라 통제 가능한 것으로 만들기 위해 존재합니다.

## Why do we use it?
*(When is it useful?)*

**EN:** Real data is never perfectly clean — phone numbers go unrecorded, revenue can legitimately be zero, price fields sometimes contain garbage text. Without `CASE WHEN` and `NULL`-handling functions, a query either crashes on the first bad row or (worse) runs to completion and quietly returns a wrong number. These tools let a query keep working *correctly* in the presence of messy, real-world data.

**KR:** 실제 데이터는 절대 완벽하게 깨끗하지 않습니다 — 전화번호가 기록되지 않거나, 매출이 정말로 0일 수 있고, 가격 필드에 이상한 텍스트가 들어있기도 합니다. `CASE WHEN`과 `NULL` 처리 함수가 없으면 쿼리는 첫 번째 잘못된 행에서 죽거나, (더 나쁘게는) 끝까지 실행되고서 조용히 틀린 숫자를 반환합니다. 이 도구들은 지저분한 실제 데이터가 있어도 쿼리가 계속 *정확하게* 동작하게 해줍니다.

## When is it used in Business Analytics?
*(Real-world use case)*

**EN:** Customer tiering ("VIP if spend ≥ 2M"), clean stakeholder-facing reports ("show '미등록' instead of a blank cell"), and margin calculations that don't explode the moment one product has zero revenue — this is some of the most frequently written BA SQL, precisely because production data is never as tidy as a tutorial's sample table.  
**KR:** 고객 등급화("지출 200만 이상이면 VIP"), 이해관계자에게 보여줄 깔끔한 리포트("빈 셀 대신 '미등록' 표시"), 상품 하나가 매출 0이라고 해서 터지지 않는 마진 계산 — 이것들은 BA가 가장 자주 작성하는 SQL 중 일부인데, 실무 데이터는 튜토리얼 샘플 테이블처럼 깔끔한 적이 절대 없기 때문입니다.

**Comparison / 비교표:**

| Task / 작업 | SQL | Pandas |
|---|---|---|
| Branch on a condition / 조건 분기 | `CASE WHEN` | `np.where()` / `np.select()` |
| Check for missing / 결측 확인 | `IS NULL` / `IS NOT NULL` | `.isna()` / `.notna()` |
| Fill missing with a default / 기본값으로 채우기 | `COALESCE` | `.fillna()` |
| Guard against divide-by-zero / 0 나눗셈 방지 | `NULLIF`, `SAFE_DIVIDE` | `.replace(0, np.nan)` |
| Range / list check / 범위·목록 확인 | `BETWEEN`, `IN` | `.between()`, `.isin()` |

---
# 📝 Syntax

## Basic Syntax
`CASE WHEN` — SQL's if/else. The simplest form picks between two outcomes.  
`CASE WHEN` — SQL의 if/else. 가장 단순한 형태는 두 결과 중 하나를 고릅니다.

In [1]:
# --- Environment setup / 환경 설정 ---
# We use DuckDB: a free, in-memory SQL engine that understands BigQuery-style syntax
# almost 1:1 (window functions, QUALIFY, ROLLUP, STRING_AGG, etc.), and can query
# pandas DataFrames directly by name -- no separate "load data" step needed.
# DuckDB는 무료 인메모리 SQL 엔진으로, BigQuery 문법(윈도우 함수, QUALIFY, ROLLUP,
# STRING_AGG 등)을 거의 그대로 이해하고, pandas DataFrame을 이름으로 바로 조회할 수
# 있습니다. 별도의 "데이터 로드" 단계가 필요 없습니다.
import duckdb
import pandas as pd
from IPython.display import display

def run(sql: str) -> pd.DataFrame:
    """Execute a SQL string against DuckDB and return the result as a DataFrame.
    SQL 문자열을 DuckDB에서 실행하고 결과를 DataFrame으로 반환합니다."""
    return duckdb.sql(sql).df()

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004],
    "amount":   [45000, 120000, 28000, 95000],
})

sql = """
SELECT
    order_id,
    amount,
    CASE
        WHEN amount >= 50000 THEN '고액'
        ELSE '소액'
    END AS amount_tier
FROM orders
"""
display(run(sql))
# pandas equivalent: np.where(df["amount"] >= 50000, "고액", "소액")


,order_id,amount,amount_tier
0,1001,45000,소액
1,1002,120000,고액
2,1003,28000,소액
3,1004,95000,고액


## Common Variations

In [2]:
# Multi-way branching: as many WHEN clauses as needed, checked top to bottom.
# The FIRST condition that matches wins -- so order matters, biggest/most-specific first.
# 다중 분기: 필요한 만큼 WHEN을 추가. 위에서부터 순서대로 검사되어
# 처음 TRUE가 되는 조건이 채택됨 -- 그래서 순서가 중요, 큰/구체적인 조건을 위로.
customers = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C04"],
    "total_spent": [1200000, 450000, 85000, 2100000],
})

sql = """
SELECT
    customer_id,
    total_spent,
    CASE
        WHEN total_spent >= 2000000 THEN 'VIP'
        WHEN total_spent >= 500000  THEN 'Gold'
        WHEN total_spent >= 100000  THEN 'Silver'
        ELSE 'Bronze'
    END AS tier
FROM customers
"""
display(run(sql))
# C01 (1,200,000) is Gold, not VIP -- it fails the >=2,000,000 check but passes >=500,000.
# pandas equivalent: np.select([cond1, cond2, cond3], [val1, val2, val3], default=val4)


,customer_id,total_spent,tier
0,C01,1200000,Gold
1,C02,450000,Silver
2,C03,85000,Bronze
3,C04,2100000,VIP


---
# 🧪 Small Examples

## Example 1 — CASE WHEN + Aggregation: Conditional Counting & Summing / 조건부 카운트·합계
**EN:** Wrapping `CASE WHEN ... THEN 1 ELSE 0 END` in `SUM()` counts how many rows meet a condition — replace `1` with a column name (like `amount`) and it sums only the matching rows' values instead. This is the fundamental building block for putting multiple conditional metrics side by side in one row of output.  
**KR:** `CASE WHEN ... THEN 1 ELSE 0 END`을 `SUM()`으로 감싸면 조건을 만족하는 행의 개수를 셀 수 있습니다 — `1`을 열 이름(예: `amount`)으로 바꾸면 매칭되는 행의 값만 합산합니다. 여러 조건부 지표를 결과 한 행에 나란히 놓는 기본 빌딩 블록입니다.

In [3]:
orders1 = pd.DataFrame({
    "order_id":    [1001, 1002, 1003, 1004, 1005, 1006],
    "status":      ["완료", "취소", "완료", "완료", "취소", "완료"],
    "amount":      [45000, 32000, 61000, 28000, 95000, 15000],
})

sql = """
SELECT
    COUNT(*) AS total_orders,
    SUM(CASE WHEN status = '완료' THEN 1 ELSE 0 END) AS completed_count,
    SUM(CASE WHEN status = '취소' THEN 1 ELSE 0 END) AS cancelled_count,
    SUM(CASE WHEN status = '완료' THEN amount ELSE 0 END) AS completed_amount
FROM orders1
"""
display(run(sql))
# pandas equivalent: (df["status"]=="완료").sum()  or  df.loc[df["status"]=="완료","amount"].sum()


,total_orders,completed_count,cancelled_count,completed_amount
0,6,4.0,2.0,149000.0


## Example 2 — IS NULL / IS NOT NULL: NULL Comparison Isn't `=` / NULL 비교는 = 이 아닌 IS
**EN:** `NULL` means "no value recorded" — it's not equal to anything, not even another `NULL`. So `WHERE phone = NULL` doesn't error, it just silently matches **zero rows**, always, regardless of the data. `NULL` has its own dedicated comparison keywords for exactly this reason.  
**KR:** `NULL`은 "기록된 값이 없음"을 뜻합니다 — 그 무엇과도, 심지어 다른 `NULL`과도 같지 않습니다. 그래서 `WHERE phone = NULL`은 오류 없이 그냥 데이터와 무관하게 항상 **0행**을 조용히 반환합니다. 바로 이런 이유로 `NULL`에는 전용 비교 키워드가 따로 있습니다.

In [4]:
customers2 = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C04"],
    "name":        ["김민수", "이영희", "박준호", "최서연"],
    "phone":       ["010-1234-5678", None, "010-9876-5432", None],
})

print("-- ❌ = NULL never matches, no matter what the data looks like --")
print("-- ❌ = NULL은 데이터와 무관하게 절대 매칭되지 않음 --")
r = run("SELECT * FROM customers2 WHERE phone = NULL")
print(f"rows: {len(r)}  (there ARE 2 customers with no phone, but this finds 0)")

print("-- ✅ IS NULL: the correct way to test for missing values --")
print("-- ✅ IS NULL: 결측값을 확인하는 올바른 방법 --")
display(run("SELECT customer_id, name FROM customers2 WHERE phone IS NULL"))

print("-- ✅ IS NOT NULL: its opposite --")
display(run("SELECT customer_id, name FROM customers2 WHERE phone IS NOT NULL"))
# pandas equivalent: IS NULL = df["phone"].isna()  |  IS NOT NULL = df["phone"].notna()


-- ❌ = NULL never matches, no matter what the data looks like --
-- ❌ = NULL은 데이터와 무관하게 절대 매칭되지 않음 --
rows: 0  (there ARE 2 customers with no phone, but this finds 0)
-- ✅ IS NULL: the correct way to test for missing values --
-- ✅ IS NULL: 결측값을 확인하는 올바른 방법 --


,customer_id,name
0,C02,이영희
1,C04,최서연


-- ✅ IS NOT NULL: its opposite --


,customer_id,name
0,C01,김민수
1,C03,박준호


## Example 3 — COALESCE: A Default Value for NULL / NULL을 기본값으로 대체
**EN:** `COALESCE(col, 'default')` returns `col`'s value if it's not `NULL`, or `'default'` otherwise — perfect for turning blank cells into something readable before a report goes out. `COALESCE` accepts more than 2 arguments too: it checks them left to right and returns the first one that isn't `NULL`, building a fallback chain.  
**KR:** `COALESCE(col, '기본값')`은 `col`이 `NULL`이 아니면 그 값을, `NULL`이면 `'기본값'`을 반환합니다 — 리포트가 나가기 전에 빈 셀을 읽을 수 있는 것으로 바꾸기에 딱 좋습니다. `COALESCE`는 인자를 3개 이상도 받을 수 있는데, 왼쪽부터 순서대로 검사해서 처음으로 `NULL`이 아닌 값을 반환하며 대체 체인을 만듭니다.

In [5]:
customers3 = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03"],
    "name":        ["김민수", "이영희", "박준호"],
    "phone":       ["010-1234-5678", None, None],
})

print("-- single fallback / 단일 기본값 --")
display(run("SELECT name, COALESCE(phone, '미등록') AS phone_display FROM customers3"))

print("-- multi-argument: mobile first, then home phone, then a text fallback --")
print("-- 여러 인자: 휴대폰 우선, 없으면 집전화, 그마저 없으면 텍스트 --")
customers3b = pd.DataFrame({
    "name":       ["김민수", "이영희", "박준호"],
    "mobile":     ["010-1111-2222", None, None],
    "home_phone": [None, "02-333-4444", None],
})
display(run("SELECT name, COALESCE(mobile, home_phone, '연락처없음') AS contact FROM customers3b"))
# pandas equivalent: df["phone"].fillna("미등록")
# multi-arg equivalent: df["mobile"].fillna(df["home_phone"]).fillna("연락처없음")


-- single fallback / 단일 기본값 --


,name,phone_display
0,김민수,010-1234-5678
1,이영희,미등록
2,박준호,미등록


-- multi-argument: mobile first, then home phone, then a text fallback --
-- 여러 인자: 휴대폰 우선, 없으면 집전화, 그마저 없으면 텍스트 --


,name,contact
0,김민수,010-1111-2222
1,이영희,02-333-4444
2,박준호,연락처없음


## Example 4 — NULLIF: Turning a Specific Value Into NULL / 특정 값을 NULL로 변환
**EN:** `NULLIF(a, b)` returns `NULL` if `a` equals `b`, otherwise returns `a` unchanged — its most common use is preventing divide-by-zero: `x / NULLIF(y, 0)` turns a `y` of `0` into `NULL` first, so the division produces a harmless `NULL` instead of a bad result.  
**KR:** `NULLIF(a, b)`는 `a`가 `b`와 같으면 `NULL`을, 아니면 `a`를 그대로 반환합니다 — 가장 흔한 용도는 0으로 나누기 방지입니다: `x / NULLIF(y, 0)`은 `y`가 `0`이면 먼저 `NULL`로 바꿔서, 나눗셈이 이상한 결과 대신 무해한 `NULL`을 내도록 만듭니다.

⚠️ **EN:** In real BigQuery, dividing by zero is a hard error that stops the whole query. In DuckDB (this notebook's engine), it doesn't error at all — it silently returns `-inf` (negative infinity), which can be *even more* dangerous, since nothing warns you and `-inf` can quietly poison any further math built on top of it. Either way, the fix is the same: guard the divisor with `NULLIF`.  
⚠️ **KR:** 실제 BigQuery에서는 0으로 나누면 하드 에러가 나서 쿼리 전체가 멈춥니다. 이 노트북의 엔진인 DuckDB는 아예 에러가 나지 않고 조용히 `-inf`(음의 무한대)를 반환하는데, 이는 아무 경고 없이 이후의 모든 계산을 조용히 오염시킬 수 있어 *더* 위험할 수 있습니다. 어느 쪽이든 해결책은 같습니다: `NULLIF`로 나누는 값을 방어하세요.

In [6]:
sales_summary = pd.DataFrame({
    "product": ["노트북", "신제품A", "마우스"],
    "revenue": [3600000, 0, 500000],   # 신제품A has zero revenue -- a brand new product / 신제품A는 매출 0인 신제품
    "cost":    [2700000, 50000, 300000],
})

print("-- ❌ unguarded: 신제품A's row silently becomes -inf, not an error --")
print("-- ❌ 방어 없음: 신제품A 행이 오류 없이 조용히 -inf가 됨 --")
display(run("SELECT product, (revenue - cost) / revenue * 100 AS margin_pct FROM sales_summary"))

print("-- ✅ guarded with NULLIF: 0 becomes NULL first, so the result is a clean NULL --")
print("-- ✅ NULLIF로 방어: 0이 먼저 NULL이 되어 결과도 깔끔한 NULL --")
display(run("SELECT product, (revenue - cost) / NULLIF(revenue, 0) * 100 AS margin_pct FROM sales_summary"))


-- ❌ unguarded: 신제품A's row silently becomes -inf, not an error --
-- ❌ 방어 없음: 신제품A 행이 오류 없이 조용히 -inf가 됨 --


,product,margin_pct
0,노트북,25.0
1,신제품A,-inf
2,마우스,40.0


-- ✅ guarded with NULLIF: 0 becomes NULL first, so the result is a clean NULL --
-- ✅ NULLIF로 방어: 0이 먼저 NULL이 되어 결과도 깔끔한 NULL --


,product,margin_pct
0,노트북,25.0
1,신제품A,NaN
2,마우스,40.0


## Example 5 — BETWEEN / IN: Range and List Conditions / 범위·목록 조건
**EN:** `BETWEEN a AND b` is shorthand for `col >= a AND col <= b` — both endpoints are included. `IN (a, b, c)` is shorthand for `col = a OR col = b OR col = c` — a match against any value in the list.  
**KR:** `BETWEEN a AND b`는 `col >= a AND col <= b`의 축약형이며, 양쪽 끝값을 모두 포함합니다. `IN (a, b, c)`는 `col = a OR col = b OR col = c`의 축약형이며, 목록 중 하나라도 일치하면 매칭됩니다.

In [7]:
products5 = pd.DataFrame({
    "product_id": ["P01","P02","P03","P04","P05","P06"],
    "name":       ["노트북","마우스","키보드","셔츠","청바지","책상"],
    "category":   ["전자","전자","전자","의류","의류","가구"],
    "price":      [1200000, 25000, 45000, 35000, 89000, 150000],
})

print("-- BETWEEN: price in [30000, 100000], both ends included --")
display(run("SELECT name, price FROM products5 WHERE price BETWEEN 30000 AND 100000"))

print("-- IN: category is one of two values -- shorter than chained ORs --")
display(run("SELECT name, category FROM products5 WHERE category IN ('전자', '의류')"))
# pandas equivalent: BETWEEN = df["price"].between(30000, 100000)  |  IN = df["category"].isin(["전자","의류"])


-- BETWEEN: price in [30000, 100000], both ends included --


,name,price
0,키보드,45000
1,셔츠,35000
2,청바지,89000


-- IN: category is one of two values -- shorter than chained ORs --


,name,category
0,노트북,전자
1,마우스,전자
2,키보드,전자
3,셔츠,의류
4,청바지,의류


## Example 6 — SAFE_DIVIDE / SAFE_CAST: BigQuery's Safe-Operation Functions / BigQuery 안전 연산
**EN:** `SAFE_DIVIDE(a, b)` is `a / NULLIF(b, 0)` compressed into one function — same idea as Example 4, spelled more explicitly. `SAFE_CAST(x AS type)` attempts a type conversion and returns `NULL` on failure instead of erroring, which matters a lot for messy external data (spreadsheet uploads, API responses) where one bad row shouldn't take down the whole query.  
**KR:** `SAFE_DIVIDE(a, b)`는 `a / NULLIF(b, 0)`을 함수 하나로 압축한 것입니다 — 예제 4와 같은 개념을 더 명시적으로 표현한 것뿐입니다. `SAFE_CAST(x AS type)`는 타입 변환을 시도하고 실패하면 오류 대신 `NULL`을 반환하는데, 외부의 지저분한 데이터(엑셀 업로드, API 응답)를 다룰 때 행 하나의 문제 때문에 쿼리 전체가 죽지 않도록 하는 데 중요합니다.

In [8]:
# DuckDB doesn't ship SAFE_DIVIDE natively, but one line adds it, so the exact BigQuery
# syntax below works unchanged. (This setup step is specific to this notebook's engine --
# in real BigQuery, SAFE_DIVIDE is already built in, no setup needed.)
# DuckDB는 SAFE_DIVIDE를 기본 제공하지 않지만, 한 줄로 추가하면 아래 BigQuery 문법이 그대로
# 동작함. (이 설정은 이 노트북의 엔진에만 필요 -- 실제 BigQuery는 이미 내장되어 있어 불필요.)
duckdb.sql("CREATE OR REPLACE MACRO safe_divide(a, b) AS CASE WHEN b = 0 THEN NULL ELSE a / b END")

print("-- SAFE_DIVIDE: same result as the NULLIF version, more explicit intent --")
sql = """
SELECT product, ROUND(SAFE_DIVIDE(revenue - cost, revenue) * 100, 1) AS margin_pct
FROM sales_summary
"""
display(run(sql))

print()
print("-- SAFE_CAST -----------------------------------------------------------")
raw_products = pd.DataFrame({
    "product":   ["노트북", "마우스", "불량데이터"],
    "price_str": ["1200000", "25000", "abc"],   # "abc" can't become a number / "abc"는 숫자가 될 수 없음
})

print("-- ❌ plain CAST: the whole query fails the moment it hits 'abc' --")
print("-- ❌ 일반 CAST: 'abc'를 만나는 순간 쿼리 전체가 실패 --")
try:
    display(run("SELECT product, CAST(price_str AS BIGINT) AS price FROM raw_products"))
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {str(e)[:120]}...")

print("-- ✅ safe cast: BigQuery's SAFE_CAST(x AS INT64) = DuckDB's TRY_CAST(x AS BIGINT) --")
print("-- ✅ 안전한 변환: BigQuery의 SAFE_CAST(x AS INT64) = DuckDB의 TRY_CAST(x AS BIGINT) --")
display(run("SELECT product, TRY_CAST(price_str AS BIGINT) AS price FROM raw_products"))
# 불량데이터's row becomes NULL instead of crashing the query -- the other 2 rows still succeed.
# 불량데이터 행은 쿼리를 죽이는 대신 NULL이 됨 -- 나머지 2행은 정상 처리.


-- SAFE_DIVIDE: same result as the NULLIF version, more explicit intent --


,product,margin_pct
0,노트북,25.0
1,신제품A,NaN
2,마우스,40.0



-- SAFE_CAST -----------------------------------------------------------
-- ❌ plain CAST: the whole query fails the moment it hits 'abc' --
-- ❌ 일반 CAST: 'abc'를 만나는 순간 쿼리 전체가 실패 --
ERROR: ConversionException: Conversion Error: Could not convert string 'abc' to INT64 when casting from source column price_str

LINE 1: SELECT prod...
-- ✅ safe cast: BigQuery's SAFE_CAST(x AS INT64) = DuckDB's TRY_CAST(x AS BIGINT) --
-- ✅ 안전한 변환: BigQuery의 SAFE_CAST(x AS INT64) = DuckDB의 TRY_CAST(x AS BIGINT) --


,product,price
0,노트북,1200000
1,마우스,25000
2,불량데이터,<NA>


## Example 7 — COUNTIF / IF: BigQuery's Conditional-Aggregation Shorthand / 조건부 집계 단축 문법
**EN:** `COUNTIF(condition)` is shorthand for `SUM(CASE WHEN condition THEN 1 ELSE 0 END)` — same result, less to type. `IF(condition, value_if_true, value_if_false)` is shorthand for `CASE WHEN condition THEN value_if_true ELSE value_if_false END`. Both are BigQuery-specific conveniences — the longer `CASE WHEN` form from Example 1 works on every SQL dialect, so use it when portability matters.  
**KR:** `COUNTIF(조건)`은 `SUM(CASE WHEN 조건 THEN 1 ELSE 0 END)`의 축약형입니다 — 결과는 같고 타이핑이 줄어듭니다. `IF(조건, 참일때값, 거짓일때값)`은 `CASE WHEN 조건 THEN 참일때값 ELSE 거짓일때값 END`의 축약형입니다. 둘 다 BigQuery 전용 편의 기능이며, 예제 1의 더 긴 `CASE WHEN` 형태는 모든 SQL dialect에서 동작하므로 이식성이 중요하면 그쪽을 쓰세요.

In [9]:
sql = """
SELECT
    COUNTIF(status = '완료') AS completed_count,
    SUM(IF(status = '완료', amount, 0)) AS completed_amount
FROM orders1
"""
display(run(sql))
# identical result to Example 1's longer CASE WHEN version -- just shorter to write.
# 예제 1의 더 긴 CASE WHEN 버전과 결과는 동일 -- 그냥 더 짧게 쓰는 것뿐.


,completed_count,completed_amount
0,4.0,149000.0


## Example 8 — Common Combinations / 자주 쓰는 조합
**EN:** **Pattern A** uses a `CASE WHEN` expression *as* the `GROUP BY` key — BigQuery lets you reuse a `SELECT` alias in `GROUP BY`, so you can bucket continuous numbers into named ranges and aggregate each bucket in one query. **Pattern B** layers `COALESCE` (clean up missing contact info) and `CASE WHEN` (assign a tier) together — a very common "make this presentable" pattern right before a report goes out the door.  
**KR:** **패턴 A**는 `CASE WHEN` 식을 그대로 `GROUP BY` 기준으로 씁니다 — BigQuery는 `SELECT` 별칭을 `GROUP BY`에서 재사용하게 해주므로, 연속된 숫자를 이름 붙은 구간으로 나누고 각 구간을 한 쿼리에서 집계할 수 있습니다. **패턴 B**는 `COALESCE`(연락처 결측 정리)와 `CASE WHEN`(등급 부여)을 함께 쌓습니다 — 리포트가 나가기 직전 "보기 좋게 만들기"의 아주 흔한 패턴입니다.

In [10]:
print("-- Pattern A: CASE WHEN as a GROUP BY bucket / CASE WHEN을 GROUP BY 기준으로 --")
orders_a = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006],
    "amount":   [45000, 120000, 28000, 95000, 210000, 15000],
})
sql_a = """
SELECT
    CASE
        WHEN amount < 50000  THEN '소액(5만 미만)'
        WHEN amount < 150000 THEN '중액(5~15만)'
        ELSE '고액(15만 이상)'
    END AS tier,
    COUNT(*) AS order_count,
    SUM(amount) AS total_amount
FROM orders_a
GROUP BY tier
ORDER BY total_amount DESC
"""
display(run(sql_a))

print("-- Pattern B: COALESCE + CASE WHEN -- a clean customer report --")
print("-- 패턴 B: COALESCE + CASE WHEN -- 깨끗한 고객 리포트 --")
customers_b = pd.DataFrame({
    "name":        ["김민수", "이영희", "박준호"],
    "phone":       ["010-1234-5678", None, None],
    "total_spent": [1200000, 450000, 85000],
})
sql_b = """
SELECT
    name,
    COALESCE(phone, '연락처 미등록') AS contact,
    CASE
        WHEN total_spent >= 500000 THEN 'Gold'
        WHEN total_spent >= 100000 THEN 'Silver'
        ELSE 'Bronze'
    END AS tier
FROM customers_b
"""
display(run(sql_b))


-- Pattern A: CASE WHEN as a GROUP BY bucket / CASE WHEN을 GROUP BY 기준으로 --


,tier,order_count,total_amount
0,중액(5~15만),2,215000.0
1,고액(15만 이상),1,210000.0
2,소액(5만 미만),3,88000.0


-- Pattern B: COALESCE + CASE WHEN -- a clean customer report --
-- 패턴 B: COALESCE + CASE WHEN -- 깨끗한 고객 리포트 --


,name,contact,tier
0,김민수,010-1234-5678,Gold
1,이영희,연락처 미등록,Silver
2,박준호,연락처 미등록,Bronze


## Example 9 — Practice / 실습 문제
**EN:** Fill in each `________` blank below, then remove the `#` in front of the matching `display(run(...))` line to check your answer. Hints: `COALESCE` `COUNTIF` `CASE` `END` `DESC`
**KR:** 아래 `________` 빈칸을 채운 뒤, 해당 `display(run(...))` 줄 앞의 `#`을 지우고 실행해서 답을 확인하세요. 힌트: `COALESCE` `COUNTIF` `CASE` `END` `DESC`

In [13]:
orders_p = pd.DataFrame({
    "order_id":      [1001, 1002, 1003, 1004, 1005],
    "customer_id":   ["C01", "C02", "C01", "C03", "C02"],
    "status":        ["완료", "취소", "완료", "완료", "완료"],
    "amount":        [45000, 32000, 0, 95000, 15000],   # order 1003 is a 0-amount promo order
    "discount_code": ["SAVE10", None, "WELCOME", None, None],
})

# Q1. Orders with no discount_code should display as '없음' (none).
# Q1. discount_code가 없는 주문은 '없음'으로 표시.
q1 = """
SELECT
    order_id,
    COALESCE(discount_code, '없음') AS discount_display
FROM orders_p
"""
display(run(q1))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

# Q2. Count of completed orders and cancelled orders, in one row (use COUNTIF).
# Q2. 완료된 주문 수와 취소된 주문 수를 한 줄로 (COUNTIF 사용).
q2 = """
SELECT
    COUNTIF(status = '완료') AS completed,
    COUNTIF(status = '취소') AS cancelled
FROM orders_p
"""
display(run(q2))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

# Q3. Bucket amount into '무료'(0), '소액'(<50000), '고액'(>=50000);
#     show count and total per bucket, sorted by total descending.
# Q3. amount를 '무료'(0원), '소액'(5만 미만), '고액'(5만 이상)으로 나누고
#     구간별 건수와 합계를 합계 내림차순으로.
q3 = """
SELECT
    CASE
        WHEN amount = 0     THEN '무료'
        WHEN amount < 50000 THEN '소액'
        ELSE '고액'
    END AS amount_tier,
    COUNT(*) AS cnt,
    SUM(amount) AS total
FROM orders_p
GROUP BY amount_tier
ORDER BY total DESC
"""
display(run(q3))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

print("✏️  Fill in the ________ blanks above, uncomment the display() lines, then re-run this cell.")
print("✏️  위 ________ 빈칸을 채우고 display() 줄의 주석을 해제한 뒤 이 셀을 다시 실행하세요.")


,order_id,discount_display
0,1001,SAVE10
1,1002,없음
2,1003,WELCOME
3,1004,없음
4,1005,없음


,completed,cancelled
0,4.0,1.0


,amount_tier,cnt,total
0,고액,1,95000.0
1,소액,3,92000.0
2,무료,1,0.0


✏️  Fill in the ________ blanks above, uncomment the display() lines, then re-run this cell.
✏️  위 ________ 빈칸을 채우고 display() 줄의 주석을 해제한 뒤 이 셀을 다시 실행하세요.


<details>
<summary>🔑 Answer / 정답 (click to expand / 클릭해서 펼치기)</summary>

```sql
-- Q1
SELECT
    order_id,
    COALESCE(discount_code, '없음') AS discount_display
FROM orders_p

-- Q2
SELECT
    COUNTIF(status = '완료') AS completed,
    COUNTIF(status = '취소') AS cancelled
FROM orders_p

-- Q3
SELECT
    CASE
        WHEN amount = 0     THEN '무료'
        WHEN amount < 50000 THEN '소액'
        ELSE '고액'
    END AS amount_tier,
    COUNT(*) AS cnt,
    SUM(amount) AS total
FROM orders_p
GROUP BY amount_tier
ORDER BY total DESC
```
</details>

---
# ⚠️ Common Mistakes

**Mistake 1 — Writing `WHERE col = NULL` instead of `WHERE col IS NULL`**
- EN: `= NULL` doesn't error, so this mistake is invisible — the query runs fine and returns zero matching rows *every single time*, regardless of how much missing data actually exists.
- KR: `= NULL`은 오류가 나지 않아서 이 실수는 눈에 띄지 않습니다 — 쿼리는 멀쩡히 실행되고, 실제로 결측 데이터가 얼마나 있든 상관없이 *항상* 0행을 반환합니다.
- ✅ Fix / 해결법: Always use `IS NULL` / `IS NOT NULL` for `NULL` comparisons — never `=` or `!=`.  
`NULL` 비교에는 항상 `IS NULL` / `IS NOT NULL`을 쓰고, `=`나 `!=`는 절대 쓰지 마세요.

**Mistake 2 — Ordering `WHEN` clauses from smallest to largest in a tier ladder**
- EN: `CASE WHEN` picks the *first* matching branch, top to bottom. If you check `total_spent >= 100000` before `total_spent >= 2000000`, a customer who spent 3,000,000 gets classified into the smaller tier, since it was checked first and already matched.
- KR: `CASE WHEN`은 위에서 아래로 검사해서 *처음* 매칭되는 분기를 선택합니다. `total_spent >= 2000000`보다 `total_spent >= 100000`을 먼저 검사하면, 300만원 쓴 고객이 먼저 매칭된 더 작은 등급으로 분류되어 버립니다.
- ✅ Fix / 해결법: Order `WHEN` clauses from the largest/most-specific threshold down to the smallest/most general.  
`WHEN` 절은 가장 크고 구체적인 기준부터 가장 작고 일반적인 기준 순서로 배치하세요.

**Mistake 3 — Dividing without guarding against a zero denominator**
- EN: A brand-new product with `revenue = 0` will crash a plain division in BigQuery outright, or (in other engines) silently produce `inf`/`-inf` that corrupts every downstream calculation touching that row.
- KR: 매출이 `0`인 신제품은 BigQuery에서 일반 나눗셈을 아예 죽여버리거나, (다른 엔진에서는) 조용히 `inf`/`-inf`를 만들어 그 행을 건드리는 이후의 모든 계산을 오염시킵니다.
- ✅ Fix / 해결법: Wrap any denominator that could be zero with `NULLIF(col, 0)`, or use `SAFE_DIVIDE` for the same protection in one function call.  
0이 될 수 있는 분모는 `NULLIF(col, 0)`으로 감싸거나, 함수 호출 하나로 같은 보호를 받는 `SAFE_DIVIDE`를 쓰세요.

**Mistake 4 — Using plain `CAST` on data you haven't validated**
- EN: `CAST(price_str AS BIGINT)` throws an error and halts the entire query the instant it meets one non-numeric string — a single bad row in a 10-million-row table can block the whole report.
- KR: `CAST(price_str AS BIGINT)`는 숫자가 아닌 문자열 하나를 만나는 순간 오류를 내며 쿼리 전체를 멈춥니다 — 천만 행짜리 테이블에서 단 하나의 잘못된 행이 리포트 전체를 막을 수 있습니다.
- ✅ Fix / 해결법: For unvalidated or external data, use `SAFE_CAST` (BigQuery) — bad values become `NULL` instead of crashing the query, and you can find and fix them afterward.  
검증되지 않았거나 외부에서 온 데이터는 `SAFE_CAST`(BigQuery)를 쓰세요 — 잘못된 값은 쿼리를 죽이는 대신 `NULL`이 되고, 나중에 찾아서 고칠 수 있습니다.

---
# 💡 Tips
Useful tips or shortcuts / 유용한 팁과 단축법

- Read a `CASE WHEN` ladder the same way you'd read a series of bouncer checks at a club door — the first condition you pass is the one that decides your fate, so put the strictest checks first.  
 `CASE WHEN` 사다리는 클럽 입구의 문지기 체크를 순서대로 통과하는 것과 같다고 생각하세요 — 처음 통과하는 조건이 운명을 결정하므로, 가장 엄격한 조건을 먼저 두세요.
- `COUNT(*)` counts rows; `COUNTIF(condition)` counts rows *matching a condition* — once you know `COUNTIF` exists, you'll reach for it constantly instead of the longer `SUM(CASE WHEN...)`.  
 `COUNT(*)`는 행을 세고, `COUNTIF(조건)`은 *조건을 만족하는* 행을 셉니다 — `COUNTIF`의 존재를 알고 나면 더 긴 `SUM(CASE WHEN...)` 대신 이걸 계속 쓰게 될 겁니다.
- Any time you write a division, pause and ask "can this denominator ever be zero?" — if the answer is "maybe," it needs `NULLIF` or `SAFE_DIVIDE`, no exceptions.  
 나눗셈을 쓸 때마다 잠깐 멈추고 "이 분모가 0이 될 수 있나?"를 물어보세요 — 답이 "그럴 수도"라면 예외 없이 `NULLIF`나 `SAFE_DIVIDE`가 필요합니다.
- `COALESCE` and `CASE WHEN` are the two tools that turn a raw, messy table into something you'd actually hand to a stakeholder — reach for both right before a query becomes "final."  
 `COALESCE`와 `CASE WHEN`은 지저분한 원본 테이블을 실제로 이해관계자에게 건넬 수 있는 것으로 바꿔주는 두 도구입니다 — 쿼리가 "최종본"이 되기 직전에 둘 다 꺼내드세요.

---
# 🔗 Related Concepts

```
SQL Learning Roadmap (this guide) / SQL 학습 로드맵 (이 가이드)
──────────────────────────────────────────────
 1. SELECT Basics
 2. Aggregation & GROUP BY
 3. JOIN
 4. Subquery & CTE
 5. Conditions & NULL Handling   ← ★ YOU ARE HERE / 지금 여기
 6. String & Date Functions
 7. Window Functions
 8. BA-Specific Patterns
```

```
The NULL-safety toolkit, matched to the problem / 문제별 NULL-안전 도구
──────────────────────────────────────────────
  Problem / 문제                       Tool / 도구
  "is this missing?"                   IS NULL / IS NOT NULL
  "값이 없는가?"
  "show a default instead of blank"    COALESCE
  "빈 값 대신 기본값 보여주기"
  "treat one value AS missing"         NULLIF
  "특정 값을 결측으로 취급"
  "don't crash on bad math/casts"      SAFE_DIVIDE / SAFE_CAST
  "잘못된 계산·변환에도 안 죽기"
```

*How is today's topic connected to other concepts?*

**EN:** Chapter 5 tools show up *inside* almost everything from earlier chapters: `CASE WHEN` inside `GROUP BY` (Ch.2), `COALESCE` after a `LEFT JOIN` to clean up the `NULL`s that JOIN deliberately introduces (Ch.3), `CASE WHEN` inside a CTE to pre-classify data before the main query touches it (Ch.4). Looking ahead, Chapter 6's string/date functions will combine constantly with `COALESCE` and `CASE WHEN` — cleaning messy text and categorizing dates are two more places "handle the exception gracefully" shows up.

**KR:** 5장의 도구들은 앞선 챕터들의 거의 모든 것 *안에* 등장합니다: `GROUP BY`(2장) 안의 `CASE WHEN`, `LEFT JOIN`(3장)이 일부러 만들어내는 `NULL`을 정리하는 `COALESCE`, 메인 쿼리가 손대기 전에 데이터를 미리 분류하는 CTE(4장) 안의 `CASE WHEN`. 앞으로 배울 6장의 문자열·날짜 함수는 `COALESCE`, `CASE WHEN`과 끊임없이 함께 쓰이는데, 지저분한 텍스트를 정리하고 날짜를 분류하는 것도 "예외를 우아하게 처리하기"가 등장하는 또 다른 자리이기 때문입니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오:**
**EN:** Ops asks: *"Can you send a clean order summary — amount tier, and whether a discount was used — that won't break if a product had zero revenue or a discount field is blank?"* This calls for `CASE WHEN` tiering plus `COALESCE` for the discount display, built defensively from the start.
**KR:** 운영팀이 묻습니다: *"주문 요약을 깔끔하게 보내줄 수 있어? 금액 등급이랑 할인 사용 여부도 같이. 매출 0이거나 할인 필드가 비어있어도 안 깨지게."* `CASE WHEN` 등급화와 `COALESCE` 할인 표시를 처음부터 방어적으로 만들면 됩니다.

**To-do / 할 일:**
- [x] Bucket each order into a 무료/소액/고액 tier with `CASE WHEN`  
`CASE WHEN`으로 각 주문을 무료/소액/고액 등급으로 나눈다
- [x] Show `'없음'` instead of a blank discount code with `COALESCE`  
`COALESCE`로 빈 할인 코드 대신 `'없음'`을 보여준다
- [x] Keep the whole thing safe even if `amount` is ever zero  
`amount`가 0이어도 전체가 안전하게 동작하도록 한다

In [12]:
orders_biz = pd.DataFrame({
    "order_id":      [4001, 4002, 4003, 4004],
    "amount":        [0, 32000, 95000, 15000],
    "discount_code": ["WELCOME", None, "SAVE10", None],
})

sql = """
SELECT
    order_id,
    amount,
    CASE
        WHEN amount = 0     THEN '무료'
        WHEN amount < 50000 THEN '소액'
        ELSE '고액'
    END AS amount_tier,
    COALESCE(discount_code, '없음') AS discount_display
FROM orders_biz
ORDER BY amount DESC
"""
display(run(sql))


,order_id,amount,amount_tier,discount_display
0,4003,95000,고액,SAVE10
1,4002,32000,소액,없음
2,4004,15000,소액,없음
3,4001,0,무료,WELCOME


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

**EN:** `CASE WHEN` branches output based on a condition, checking each `WHEN` top-to-bottom and stopping at the first match — which is why tier ladders must go from most-specific to least-specific. `NULL` represents missing data and can't be tested with `=`; `IS NULL`/`IS NOT NULL` are the only correct comparisons. `COALESCE` supplies a default (or chain of fallbacks) when a value is `NULL`, while `NULLIF` does the reverse — turning a specific value (often `0`, to guard a division) into `NULL` on purpose. `SAFE_DIVIDE` and `SAFE_CAST` are BigQuery's built-in versions of these same NULL-guarding patterns, returning `NULL` instead of crashing when the underlying operation would fail. `COUNTIF` and `IF` are shorter BigQuery-only spellings of the `CASE WHEN` patterns from Example 1.

**KR:** `CASE WHEN`은 조건에 따라 결과를 분기하며, 각 `WHEN`을 위에서 아래로 검사해서 처음 매칭되는 곳에서 멈춥니다 — 그래서 등급 사다리는 가장 구체적인 것부터 가장 일반적인 것 순서로 가야 합니다. `NULL`은 결측 데이터를 나타내며 `=`로 검사할 수 없고, `IS NULL`/`IS NOT NULL`만이 올바른 비교입니다. `COALESCE`는 값이 `NULL`일 때 기본값(또는 대체 체인)을 제공하고, `NULLIF`는 그 반대로 특정 값(주로 나눗셈을 방어하기 위한 `0`)을 의도적으로 `NULL`로 바꿉니다. `SAFE_DIVIDE`와 `SAFE_CAST`는 이 같은 NULL 방어 패턴을 BigQuery가 내장 함수로 제공한 것으로, 연산이 실패할 상황에서 죽는 대신 `NULL`을 반환합니다. `COUNTIF`와 `IF`는 예제 1의 `CASE WHEN` 패턴을 더 짧게 쓰는 BigQuery 전용 표기법입니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence. / 오늘 배운 내용을 한 문장으로.

> **EN:** `CASE WHEN` is SQL's if/else and `NULL` is SQL's "unknown" — and once you know `NULL` breaks `=` but respects `IS NULL`/`COALESCE`/`NULLIF`, every "safe" function in this chapter (`SAFE_DIVIDE`, `SAFE_CAST`, `COUNTIF`) is just that same NULL-handling idea, wearing a shorter name.

> **KR:** `CASE WHEN`은 SQL의 if/else이고 `NULL`은 SQL의 "알 수 없음"입니다 — `NULL`이 `=`를 깨뜨리지만 `IS NULL`/`COALESCE`/`NULLIF`는 존중한다는 것을 알고 나면, 이번 챕터의 모든 "안전한" 함수(`SAFE_DIVIDE`, `SAFE_CAST`, `COUNTIF`)는 같은 NULL 처리 개념이 더 짧은 이름을 쓴 것일 뿐임을 알게 됩니다.

---
# ❓ Review Questions

**Q1.** In a `CASE WHEN` tier ladder with `>= 2000000`, `>= 500000`, `>= 100000`, `ELSE`, why must the conditions be written in that exact order (largest first)?
**Q1.** `CASE WHEN` 등급 사다리에서 `>= 2000000`, `>= 500000`, `>= 100000`, `ELSE` 조건은 왜 반드시 그 순서(큰 것부터)로 작성해야 하는가?

**Q2.** `WHERE phone = NULL` runs without any error but always returns zero rows. Why doesn't it error, and what should you write instead?
**Q2.** `WHERE phone = NULL`은 오류 없이 실행되지만 항상 0행을 반환한다. 왜 오류가 나지 않으며, 대신 무엇을 써야 하는가?

**Q3.** What's the difference between what `COALESCE` does and what `NULLIF` does — could you describe them as opposites?
**Q3.** `COALESCE`가 하는 일과 `NULLIF`가 하는 일의 차이는 무엇인가 — 둘을 서로 반대라고 설명할 수 있는가?

**Q4.** Why does `(revenue - cost) / revenue` need protection when `revenue` can be `0`, and name two different ways to add that protection.
**Q4.** `revenue`가 `0`이 될 수 있을 때 `(revenue - cost) / revenue`는 왜 보호가 필요하며, 그 보호를 추가하는 서로 다른 두 가지 방법을 말해보라.

**Q5.** `COUNTIF(status = '완료')` and `SUM(CASE WHEN status = '완료' THEN 1 ELSE 0 END)` return the same number. What's the actual difference between them, and when would you choose the longer form?
**Q5.** `COUNTIF(status = '완료')`와 `SUM(CASE WHEN status = '완료' THEN 1 ELSE 0 END)`는 같은 숫자를 반환한다. 둘의 실질적인 차이는 무엇이며, 언제 더 긴 형태를 선택해야 하는가?

---
*📅 Try answering these again in a few days. / 며칠 후 다시 답해보세요.*